# AMBER RAG LLM Support System

In [9]:
from importlib import reload  # Reload modules during development
import os  # OS utilities
import requests  # HTTP requests
import numpy as np  # Numerical operations
import faiss  # Vector similarity search

import database  # Local database module
from database import AmberChromaAPI  # Amber-Chroma interface

from pypdf import PdfReader  # PDF reading
from sentence_transformers import SentenceTransformer  # Text embeddings

reload(database)  # Refresh module changes


<module 'database' from '/home/bsauce11/RAG_Prototype/Code_Saucedo/Main_PreDev/database.py'>

In [6]:
"""
RAG Pipeline Execution

Steps:
1. Retrieve relevant chunks (ChromaDB + optional PDF).
2. Build structured context for the LLM.
3. Format messages into chat-style prompt.
4. Generate final response using LLaMA (via Ollama).
5. Print the final answer.
"""

# Retrieve relevant chunks from hybrid retriever 
chunks = retrieve_with_pdf(
    question,
    k_chroma=50,     # Number of ChromaDB results
    k_pdf=5,         # Number of PDF results
    threshold=threshold_ChromaDB    # Similarity threshold
)

# Build LLM-ready context and prompt 
context = build_context(chunks)          # Combine chunks into single context block
messages = build_prompt(question, context)  # Create chat-style messages

# Generate final answer using local LLaMA 
answer = generate_llama(messages)  # Send prompt to Ollama LLaMA model




NameError: name 'retrieve_with_pdf' is not defined

In [7]:
# Chroma DB instance
API_CHROMA_DB = AmberChromaAPI(db_path="/opt/chromadb/data/amber_chroma_db")
# Embedding model
EMBEDDER = SentenceTransformer("all-MiniLM-L6-v2")
# PDF file path
PDF_ADDRESS = "Amber25.pdf"
# Local Ollama server URL
OLLAMA_URL = "http://127.0.0.1:11434"


Using local ChromaDB path: /opt/chromadb/data/amber_chroma_db


In [8]:
threshold_ChromaDB = 0.35 # Similarity threshold for Mails
threshold_PDF = 0.45  # Similarity threshold for PDF results

In [ ]:
import pprint  # Pretty-printing for debugging
def retrieve(question: str, k: int = 100, threshold: float = 0.2, where=None):
    """
    Uses your api.query_embeddings to fetch candidates, then sorts & returns top-k.
    Expected item keys (from your method): embedding, similarity, metadata, document
    """
    # Query the vector database for similar embeddings
    results = API_CHROMA_DB.query(question, n=100, threshold=threshold, where=where)

    # Sort results by similarity score (highest first) and take top-k
    results = sorted(results, key=lambda r: r["similarity"], reverse=True)[:k]

    # Normalize results into a clean, consistent structure
    out = []
    for r in results:
        meta = r.get("metadata", {}) or {}
        out.append({
            "text": r.get("documents", ""),
            "author": meta.get("author", "Unknown"),
            "subject": meta.get("subject", "(no subject)"),
            "date_iso": meta.get("date_iso", ""),
            "similarity": float(r.get("similarity", 0.0)),
        })
    return out

In [ ]:
# Chunking
def chunk_text(text, size=700, overlap=100):
    """
    Splits long text into overlapping chunks.

    Parameters:
        text (str): Full document text.
        size (int): Maximum characters per chunk.
        overlap (int): Number of overlapping characters between chunks.

    Returns:
        list[str]: List of text chunks.
    """
    # Store generated chunks
    chunks = []

    # Step size to maintain overlap
    step = size - overlap

    # Iterate over text using sliding window
    for i in range(0, len(text), step):
        # Extract chunk of fixed size
        chunk = text[i:i + size]
        # Skip empty/whitespace chunks
        if chunk.strip():
            chunks.append(chunk)

    # Return all chunks
    return chunks


# Build or Load FAISS Index
def build_or_load_index(pdf_path, chunk_size=700, overlap=100):
    """
    Builds a FAISS vector index from a PDF file or loads an existing one.

    Steps:
    1. Check if FAISS index already exists.
    2. If yes → load index and chunks.
    3. If no → extract text from PDF.
    4. Chunk text into overlapping pieces.
    5. Generate embeddings.
    6. Normalize embeddings for cosine similarity.
    7. Build FAISS index and save to disk.

    Returns:
        index (faiss.Index): FAISS search index.
        chunks (list[str]): Corresponding text chunks.
    """

    # Remove .pdf extension
    base_name = os.path.splitext(pdf_path)[0]
    # Path to saved FAISS index
    index_path = base_name + ".faiss"
    # Path to saved chunks
    chunks_path = base_name + ".chunks.npy"

    # Load if already exists
    if os.path.exists(index_path) and os.path.exists(chunks_path):
        # Load FAISS index
        index = faiss.read_index(index_path)
        # Load chunks
        chunks = np.load(chunks_path, allow_pickle=True)
        # Return loaded objects
        return index, chunks

    # If no saved index → build new one
    reader = PdfReader(pdf_path)
    # Store extracted text
    text = ""

    # Loop through all pages
    for i, page in enumerate(reader.pages):
        try:
            # Extract page text
            extracted = page.extract_text()
            # Append to full text
            if extracted:
                text += extracted + "\n"
        except Exception:
            print(f"Skipped unreadable page {i}")

    # Split text into overlapping chunks
    chunks = chunk_text(text, chunk_size, overlap)

    # Generate embeddings for all chunks
    embeddings = EMBEDDER.encode(
        chunks,
        batch_size=32,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    embeddings = np.asarray(embeddings, dtype=np.float32)
    embeddings = np.ascontiguousarray(embeddings)

    # Normalize embeddings for cosine similarity search
    faiss.normalize_L2(embeddings)

    # Get embedding dimension
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    # Save index and chunks to disk
    faiss.write_index(index, index_path)
    np.save(chunks_path, chunks)

    return index, chunks



# Search Function
def search_pdf(pdf_path, query, top_k=5, threshold=threshold_PDF):
    """
    Searches a PDF using FAISS similarity search.

    Steps:
    1. Load or build FAISS index.
    2. Embed the query.
    3. Normalize query embedding.
    4. Perform similarity search.
    5. Return top-k most similar chunks.

    Returns:
        list[dict]: List of matching chunks with similarity scores.
    """

    index, chunks = build_or_load_index(pdf_path)

    # Encode query into embedding
    query_embedding = EMBEDDER.encode(
        [query],
        convert_to_numpy=True
    )

    # Normalize for cosine similarity
    query_embedding = np.asarray(query_embedding, dtype=np.float32)
    query_embedding = np.ascontiguousarray(query_embedding)

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, min(top_k, len(chunks)))

    results = []

    for rank, idx in enumerate(indices[0], 1):
        similarity = float(scores[0][rank - 1])
        chunk_text = chunks[idx]

        if similarity < threshold:
            continue

        results.append({
            "text": chunk_text,
            "similarity": similarity
        })

    return results


In [ ]:
SYSTEM_PROMPT = """
You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).


CORE RULES-
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention an Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.
7) Do NOT include citation markers, chunk labels, similarity scores,
   reference numbers, or metadata in your output.
8) Do NOT repeat metadata from the Context.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Output only the final answer.
Do not include citations, references, or metadata.
"""


In [ ]:
def build_context(chunks, max_chars: int = 9000):
    """
    Builds a single context string from retrieved chunks for LLM input.

    Steps:
    1. Iterate through retrieved chunks (already ranked).
    2. Add citation-style headers for traceability.
    3. Concatenate chunk text with metadata.
    4. Stop when max character limit is reached.
    5. Return a formatted context block.

    Parameters:
        chunks (list[dict]): Retrieved documents with metadata.
        max_chars (int): Maximum total characters allowed.

    Returns:
        str: Combined context string ready for LLM.
    """
    parts, used = [], 0

    # Loop through ranked chunks
    for i, ch in enumerate(chunks, 1):
        # Create citation-style header with metadata
        header = (
            f"[CITE {i}] {ch['subject']} | {ch['author']} "
            f"(similarity={ch['similarity']:.3f})"
        )
        body = (ch["text"] or "").strip()
        block = header + "\n" + body + "\n"
        # Stop if adding this block exceeds max character limit
        if used + len(block) > max_chars:
            break
        parts.append(block)
        used += len(block)

    # Join all blocks with separator for readability
    return "\n\n-----\n\n".join(parts)

In [ ]:
def build_prompt(question: str, context: str) -> list:
    """
    Builds a structured chat prompt for a chat-based LLM.

    Steps:
    1. Add system-level instructions (SYSTEM_PROMPT).
    2. Provide retrieved context to ground the answer.
    3. Append the user’s question.
    4. Instruct the model to answer with citations.

    Parameters:
        question (str): User's question.
        context (str): Retrieved contextual information.

    Returns:
        list[dict]: Chat-formatted messages for LLM API.
    """

    return [
        # System message defines assistant behavior
        {"role": "system", "content": SYSTEM_PROMPT},

        # User message contains context + question
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}\n\n"
                "Answer"
            )
        }
    ]


In [ ]:
def generate_llama(messages, model="llama3", temperature=0.2):
    """
    Sends chat messages to a local LLaMA 3 model via Ollama API.

    Steps:
    1. Convert structured chat messages into a single prompt string.
    2. Format roles explicitly (System/User/Assistant).
    3. Send POST request to Ollama's /api/generate endpoint.
    4. Return the generated response text.

    Parameters:
        messages (list[dict]): Chat-style messages (role + content).
        model (str): Ollama model name.
        temperature (float): Controls randomness of generation.

    Returns:
        str: Model-generated response.
    """

    # Convert chat-style messages into plain text prompt
    prompt = ""

    for msg in messages:
        role = msg["role"]        # Message role (system/user/assistant)
        content = msg["content"]  # Message content

        # Format based on role
        if role == "system":
            prompt += f"System: {content}\n"
        elif role == "user":
            prompt += f"User: {content}\n"
        elif role == "assistant":
            prompt += f"Assistant: {content}\n"

    # Signal the model to generate assistant continuation
    prompt += "Assistant:"

    # Send request to local Ollama server
    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": model,          # Model name (e.g., llama3)
            "prompt": prompt,        # Fully constructed prompt
            "temperature": temperature,  # Sampling randomness
            "stream": False          # Disable streaming response
        }
    )

    response.raise_for_status()  # Raise error if request failed

    # Return clean generated response text
    return response.json()["response"].strip()


In [ ]:
def retrieve(question: str, k: int = 50, threshold: float = 0.2, where=None):

    results = API_CHROMA_DB.query(
        text=question,
        n=k,
        where=where,
        threshold=threshold
    )

    documents = results["documents"]
    metadatas = results["metadatas"]
    scores = results["scores"]

    out = []

    for doc, meta, score in zip(documents, metadatas, scores):
        out.append({
            "text": doc,
            "author": meta.get("author", "Unknown"),
            "subject": meta.get("subject", "(no subject)"),
            "date_iso": meta.get("date_iso", ""),
            "similarity": score,
        })

    return out


### RAG Pipeline Execution


## User Query

In [ ]:
# User Prompt / Question
question = """
I have been trying to find a suitable OS for the server machine that I have where I can install Amber and Schrodinger licenses together.

My machine currently is:
Rocky Linux 8.9 (Green Obsidian)
CPE OS Name: cpe:/o:rocky:rocky:8:GA
Kernel: Linux 4.18.0-513.9.1.el8_9.x86_64
The GPU is: NVIDIA RTX 3070, 5888 CUDA Cores, 8 GB GDDR6 Memory, PCIe 4.0 GPU

The Machine could launch Schrodinger successfully, but I could not get pmem.cuda installed (It seems that the compiling of cuda11 at that OS does not work well?).

Do you recommend moving into UBUNTU? If you have any recommendations, I would be grateful if you provide some details.
"""

### LLM Response

In [ ]:
print(answer)